### Workshop the steps in a synthetic reconstruction of narG variants. 

First, build two faa's, one to be used as a reference, the other to generate raw reads.

In [ ]:
ref_directory = '../data/whole_genomes/K00370_rep.faa'

from Bio import SeqIO
import random

# Read all sequences
records = list(SeqIO.parse(ref_directory, "fasta"))
print(f"Total sequences: {len(records)}")

# Shuffle and split 70/30
random.shuffle(records)
split_point = int(0.89 * len(records)) #0.89 makes it so there are exactly 100 spike-in sequences

reference_set = records[:split_point]
spikein_set = records[split_point:]

# Write outputs
SeqIO.write(reference_set, "../out/reconstruction/reference.faa", "fasta")
SeqIO.write(spikein_set, "../out/reconstruction/spikein.faa", "fasta")

print(f"Reference: {len(reference_set)} sequences")
print(f"Spike-in: {len(spikein_set)} sequences")

Total sequences: 906
Reference: 806 sequences
Spike-in: 100 sequences


Next, generate reads. 

In [8]:
from Bio import SeqIO
import random

def back_translate(protein_seq):
    """Convert protein sequence to DNA using common codons"""
    codon_table = {
        'A': 'GCT', 'R': 'CGT', 'N': 'AAT', 'D': 'GAT',
        'C': 'TGT', 'Q': 'CAG', 'E': 'GAA', 'G': 'GGT',
        'H': 'CAT', 'I': 'ATT', 'L': 'CTG', 'K': 'AAA',
        'M': 'ATG', 'F': 'TTT', 'P': 'CCT', 'S': 'TCT',
        'T': 'ACT', 'W': 'TGG', 'Y': 'TAT', 'V': 'GTT',
        '*': 'TAA'
    }
    dna_seq = ''
    for aa in protein_seq:
        dna_seq += codon_table.get(aa, 'NNN')  # Use NNN for unknown amino acids
    return dna_seq

def generate_paired_end_reads(dna_sequence, read_length=150, coverage=30, insert_size=300):
    """Generate paired-end reads with proper insert size"""
    reads_r1 = []
    reads_r2 = []
    seq_length = len(dna_sequence)
    
    # Calculate number of read pairs needed
    num_pairs = (seq_length * coverage) // (2 * read_length)
    
    for _ in range(num_pairs):
        # Random start position for R1
        start_r1 = random.randint(0, seq_length - insert_size)
        end_r1 = start_r1 + read_length
        start_r2 = start_r1 + insert_size - read_length
        
        # Ensure we don't go beyond sequence boundaries
        if start_r2 + read_length <= seq_length:
            read_r1 = dna_sequence[start_r1:end_r1]
            read_r2 = dna_sequence[start_r2:start_r2 + read_length]
            
            # R2 is reverse complement (simulating actual sequencing)
            read_r2 = reverse_complement(read_r2)
            
            reads_r1.append(read_r1)
            reads_r2.append(read_r2)
    
    return reads_r1, reads_r2

def reverse_complement(seq):
    """Simple reverse complement"""
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(comp.get(base, base) for base in reversed(seq))

# Read spike-in protein sequences
spikein_records = list(SeqIO.parse("../out/reconstruction/spikein.faa", "fasta"))

# Generate paired-end DNA reads
all_r1 = []
all_r2 = []

for record in spikein_records:
    # Convert protein to DNA first!
    dna_sequence = back_translate(str(record.seq))
    print(f"Converted {record.id}: {len(record.seq)}aa -> {len(dna_sequence)}bp DNA")
    
    r1, r2 = generate_paired_end_reads(dna_sequence, read_length=150, coverage=15)
    all_r1.extend(r1)
    all_r2.extend(r2)

print(f"Generated {len(all_r1)} DNA read pairs")

# Write R1 and R2 files as FASTA
with open("../out/reconstruction/spikein_DNA_R1.fasta", "w") as f1, \
     open("../out/reconstruction/spikein_DNA_R2.fasta", "w") as f2:
    
    for i, (read1, read2) in enumerate(zip(all_r1, all_r2)):
        # R1 file - FASTA format
        f1.write(f">read_{i}/1\n")
        f1.write(f"{read1}\n")
        
        # R2 file - FASTA format  
        f2.write(f">read_{i}/2\n")
        f2.write(f"{read2}\n")

print("Paired-end DNA reads written to:")
print("  ../out/reconstruction/spikein_DNA_R1.fasta")
print("  ../out/reconstruction/spikein_DNA_R2.fasta")

Converted cry:B7495_10885: 1225aa -> 3675bp DNA
Converted adp:NCTC12871_01270: 1250aa -> 3750bp DNA
Converted slv:SLIV_13640: 911aa -> 2733bp DNA
Converted tni:TVNIR_1119: 1167aa -> 3501bp DNA
Converted ail:FLP10_05130: 1248aa -> 3744bp DNA
Converted ecos:EC958_1735: 1287aa -> 3861bp DNA
Converted asez:H9L21_04190: 1232aa -> 3696bp DNA
Converted malu:KU6B_34520: 472aa -> 1416bp DNA
Converted cauf:CSW63_03735: 1242aa -> 3726bp DNA
Converted gpb:HDN1F_13240: 1250aa -> 3750bp DNA
Converted sule:GFS03_05630: 1173aa -> 3519bp DNA
Converted afk:ACNAN0_11500: 1223aa -> 3669bp DNA
Converted omo:RHO11_02430: 1235aa -> 3705bp DNA
Converted sml:Smlt2774: 1268aa -> 3804bp DNA
Converted npe:Natpe_3622: 1179aa -> 3537bp DNA
Converted pde:Pden_4236: 1254aa -> 3762bp DNA
Converted nmv:NITMOv2_0255: 1145aa -> 3435bp DNA
Converted psi:S70_07650: 1253aa -> 3759bp DNA
Converted pew:KZJ38_17505: 1285aa -> 3855bp DNA
Converted mshj:MSHI_28400: 1225aa -> 3675bp DNA
Converted lrf:LAR_0939: 1221aa -> 3663bp DN